In [1]:
!nvidia-smi


Tue Sep  2 07:55:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.0/97.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.7/142.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.6/413.6 kB 27.8 MB/s eta 0:00:00


In [3]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
  Successfully uninstalled transformers-4.55.4
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4


In [4]:
import os, numpy as np, random
from datasets import load_dataset, interleave_datasets

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          EarlyStoppingCallback)

import torch
import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [5]:
# Works on latest 'datasets' (no scripts). Colab-ready.

!pip install -q datasets

import os, tarfile, requests
from datasets import load_dataset

def fetch_and_extract_xlsum(lang, version="2.0"):
    """
    Downloads {lang}_XLSum_v{version}.tar.bz2 from HF and extracts
    {lang}_train.jsonl / {lang}_val.jsonl / {lang}_test.jsonl into /content/xlsum/{lang}/
    Returns a dict-style HuggingFace DatasetDict with splits.
    """
    lang_dir = f"/content/xlsum/{lang}"
    os.makedirs(lang_dir, exist_ok=True)

    url = f"https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/data/{lang}_XLSum_v{version}.tar.bz2"
    tar_path = f"{lang_dir}/{lang}_XLSum_v{version}.tar.bz2"

    # download if needed (streaming to file)
    if not os.path.exists(tar_path):
        print(f"Downloading {lang}…")
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(tar_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)

    # extract if not yet extracted
    expected_files = [
        f"{lang}_train.jsonl",
        f"{lang}_val.jsonl",
        f"{lang}_test.jsonl",
    ]
    if not all(os.path.exists(os.path.join(lang_dir, fn)) for fn in expected_files):
        print(f"Extracting {lang}…")
        with tarfile.open(tar_path, "r:bz2") as tf:
            tf.extractall(lang_dir)

    # now load with the generic JSON builder (no scripts)
    data_files = {
        "train":      os.path.join(lang_dir, f"{lang}_train.jsonl"),
        "validation": os.path.join(lang_dir, f"{lang}_val.jsonl"),
        "test":       os.path.join(lang_dir, f"{lang}_test.jsonl"),
    }
    ds = load_dataset("json", data_files=data_files)
    return ds

# ---- pick your languages
en = fetch_and_extract_xlsum("english")   # columns: id, url, title, summary, text


print(en)
print(en["train"][0].keys())
fr = fetch_and_extract_xlsum("french")
hi = fetch_and_extract_xlsum("hindi")

print(fr, hi)

Extracting english…


/tmp/ipython-input-1356644993.py:39: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(lang_dir)


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

...

Extracting hindi…


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 8697
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 1086
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 1086
    })
}) DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 70778
    })
    validation: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 8847
    })
    test: Dataset({
        features: ['id', 'url', 'title', 'summary', 'text'],
        num_rows: 8847
    })
})


In [6]:
from datasets import interleave_datasets

SEED = 42
SAMPLES_TRAIN = 3000   # raise later when stable
SAMPLES_VAL   = 300
SAMPLES_TEST  = 50

def with_lang_column(dset, code):
    return dset.add_column("lang", [code]*len(dset))

parts_tr = [
    with_lang_column(en["train"].shuffle(seed=SEED).select(range(min(SAMPLES_TRAIN, len(en["train"])))), "en"),
    with_lang_column(fr["train"].shuffle(seed=SEED).select(range(min(SAMPLES_TRAIN, len(fr["train"])))), "fr"),
    with_lang_column(hi["train"].shuffle(seed=SEED).select(range(min(SAMPLES_TRAIN, len(hi["train"])))), "hi"),
]
parts_va = [
    with_lang_column(en["validation"].shuffle(seed=SEED).select(range(min(SAMPLES_VAL, len(en["validation"])))), "en"),
    with_lang_column(fr["validation"].shuffle(seed=SEED).select(range(min(SAMPLES_VAL, len(fr["validation"])))), "fr"),
    with_lang_column(hi["validation"].shuffle(seed=SEED).select(range(min(SAMPLES_VAL, len(hi["validation"])))), "hi"),
]

parts_tst = [
    with_lang_column(en["test"].shuffle(seed=SEED).select(range(min(SAMPLES_VAL, len(en["test"])))), "en"),
    with_lang_column(fr["test"].shuffle(seed=SEED).select(range(min(SAMPLES_VAL, len(fr["test"])))), "fr"),
    with_lang_column(hi["test"].shuffle(seed=SEED).select(range(min(SAMPLES_VAL, len(hi["test"])))), "hi"),
]

train_mix = interleave_datasets(parts_tr, seed=SEED)
val_mix   = interleave_datasets(parts_va, seed=SEED)
test_mix   = interleave_datasets(parts_tst, seed=SEED)


train_mix, val_mix, test_mix


Flattening the indices:   0%|          | 0/3000 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/3000 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/3000 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/300 [00:00<?, ? examples/s]

(Dataset({
     features: ['id', 'url', 'title', 'summary', 'text', 'lang'],
     num_rows: 9000
 }),
 Dataset({
     features: ['id', 'url', 'title', 'summary', 'text', 'lang'],
     num_rows: 900
 }),
 Dataset({
     features: ['id', 'url', 'title', 'summary', 'text', 'lang'],
     num_rows: 900
 }))

In [7]:
print(f"Features: {train_mix.column_names}")

Features: ['id', 'url', 'title', 'summary', 'text', 'lang']


In [8]:



print(f"Features: {train_mix.column_names}")
print("\text:")

print(test_mix[1]["text"])

print("\nSummary:")

print(test_mix[1]["summary"])

Features: ['id', 'url', 'title', 'summary', 'text', 'lang']
	ext:
Michelle Obama "Donald Trump n'est pas le bon président pour notre pays", a déclaré l'ancienne première dame des États-Unis dans un message enregistré diffusé lors de la convention des démocrates. Les membres du parti républicain de M. Trump, qui ont été mécontents, l'ont également interpellé lors de la conférence du parti démocrate. L'élection aura lieu le mardi 3 novembre. En raison de l'épidémie de coronavirus, les démocrates ont abandonné les plans d'une fête extravagante avec des lâchers de ballons à Milwaukee, Wisconsin. Mais il n'est pas certain que le programme largement virtuel de discours préenregistrés sans public en direct puisse générer le même niveau d'enthousiasme que les rassemblements pré-covid19 des fidèles du parti. Les républicains seront confrontés au même défi lorsqu'ils présenteront leurs arguments pendant quatre années supplémentaires à la Maison Blanche lors d'une convention réduite de manière dr

In [9]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [10]:
model = "csebuetnlp/mT5_multilingual_XLSum"

tokenizer = AutoTokenizer.from_pretrained(model)  #load a tokenizer

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

In [11]:
from datasets import DatasetDict

# assuming you already have these as `datasets.Dataset` objects:
# train_mix, val_mix, test_mix

data_splits = DatasetDict({
    "train": train_mix,
    "validation": val_mix,
    "test": test_mix,
})

In [12]:
data_splits['test'][0]['text']

'Unite said it would "use all its might" while the GMB said the outlook, confirmed at a meeting, was a "real kick in the teeth". Unless new business comes in, the south Wales site could be left in a worse case scenario with only 600 workers by 2021. The prime minister said a "dialogue" would continue with Ford about help. On Wednesday, Ford shared its five-year outlook with unions in the wake of reductions in planned investment in its new Dragon engine. It said there were "healthy volumes" of work over the next two to three years. "Beyond that, identified workload is reduced and whilst such a forecast is not unusual, given the cyclical nature of our business, it is a concern and we fully understand that," said the company. Analysis: Ford Bridgend loses out in global race As it happened: Ford Bridgend jobs Speaking at Prime Minister\'s Question Time, Theresa May told Parliament: "Ford is an important investor here. It has been established here for over 100 years. We now account for arou

In [13]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['text'] , max_length = 1024, truncation = True )

    with tokenizer.as_target_tokenizer():
        target_encodings = tokenizer(example_batch['summary'], max_length = 128, truncation = True )

    return {
        'input_ids' : input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }


In [14]:
dataset_splits_pt = data_splits.map(convert_examples_to_features, batched = True)

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [15]:
dataset_splits_pt['test']

Dataset({
    features: ['id', 'url', 'title', 'summary', 'text', 'lang', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 900
})

In [16]:
dataset_splits_pt['train']

Dataset({
    features: ['id', 'url', 'title', 'summary', 'text', 'lang', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 9000
})

In [17]:
# Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [22]:
from transformers import TrainingArguments, Trainer


trainer_args = TrainingArguments(
    output_dir="model_mtf-final",
    num_train_epochs=2,                   # a bit more training than 2
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,       # effective batch = 16
    eval_strategy="steps",          # <-- correct key
    eval_steps=450,                       # more frequent feedback
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    logging_steps=25,
    warmup_steps=200,                     # okay w/ Adafactor's relative schedule
    weight_decay=0.01,
    optim="adafactor",                    # memory saver on MPS
    dataloader_pin_memory=False,          # MPS/CPU
    gradient_checkpointing=True,          # big memory win
    report_to="none"
)


trainer = Trainer(model=model_pegasus, args=trainer_args,
                  processing_class=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_splits_pt["train"],
                  eval_dataset=dataset_splits_pt["validation"])

In [23]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss
450,1.605300,1.638976
900,1.498600,1.646511


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:4034: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 84, 'num_beams': 4, 'length_penalty': 0.6, 'no_repeat_ngram_size': 2}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Step,Training Loss,Validation Loss
450,1.605300,1.638976
900,1.498600,1.646511


TrainOutput(global_step=1126, training_loss=1.5750353965200286, metrics={'train_runtime': 7578.5497, 'train_samples_per_second': 2.375, 'train_steps_per_second': 0.149, 'total_flos': 2.70718621595136e+16, 'train_loss': 1.5750353965200286, 'epoch': 2.0})

In [26]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [27]:
import evaluate

rouge_metric = evaluate.load('rouge')
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
#rouge_metric = load_metric('rouge')

In [33]:
model_pegasus.to(device).eval()

test10 = dataset_splits_pt["test"].select(range(10))
rouge = evaluate.load("rouge")

# dynamic padding collator
collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus, padding=True, return_tensors="pt")

preds, refs = [], []
batch_size = 2

for i in range(0, len(test10), batch_size):
    batch = test10[i:i+batch_size]

    # Build features for the collator from your pretokenized columns
    features = [
        {"input_ids": ids, "attention_mask": mask}
        for ids, mask in zip(batch["input_ids"], batch["attention_mask"])
    ]
    batch_pt = collator(features)  # pads to the longest in this batch
    batch_pt = {k: v.to(device) for k, v in batch_pt.items() if k in ("input_ids", "attention_mask")}

    with torch.no_grad():
        out = model_pegasus.generate(
            **batch_pt,
            max_new_tokens=128,
            min_new_tokens=32,
            num_beams=4,
            length_penalty=0.8,
            do_sample=False
        )

    decoded = tokenizer.batch_decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    preds.extend([d.strip() for d in decoded])
    refs.extend([r.strip() for r in batch["summary"]])

print(rouge.compute(predictions=preds, references=refs, use_stemmer=True))

{'rouge1': np.float64(0.22361415815886343), 'rouge2': np.float64(0.05426129426129426), 'rougeL': np.float64(0.13631580845774324), 'rougeLsum': np.float64(0.13638599174169985)}


In [34]:
from tqdm.auto import tqdm

test10 = dataset_splits_pt["test"].select(range(10))
SRC_COL, REF_COL = "text", "summary"
preds = []

for i in range(0, len(test10), 2):
    batch = test10[i : i + 2]
    enc = tokenizer(batch[SRC_COL], truncation=True, max_length=1024,
                    padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model_pegasus.generate(
            **enc, max_new_tokens=128, min_new_tokens=32,
            num_beams=4, length_penalty=0.8, do_sample=False
        )
    preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))

for idx, (src, ref, pred) in enumerate(zip(test10[SRC_COL], test10[REF_COL], preds)):
    print(f"\n--- Example {idx} ---")
    print("SOURCE    :", (src[:400] + "...") if len(src) > 400 else src)
    print("REFERENCE :", ref)
    print("PREDICTION:", pred.strip())



--- Example 0 ---
SOURCE    : Unite said it would "use all its might" while the GMB said the outlook, confirmed at a meeting, was a "real kick in the teeth". Unless new business comes in, the south Wales site could be left in a worse case scenario with only 600 workers by 2021. The prime minister said a "dialogue" would continue with Ford about help. On Wednesday, Ford shared its five-year outlook with unions in the wake of re...
REFERENCE : Unions have said they would fight a potential 1,160 job losses at Ford's engine plant in Bridgend.
PREDICTION: The future of Ford's Bridgend engine plant is a "nightmare", union leaders have said, amid concerns over the long-term future.

--- Example 1 ---
SOURCE    : Michelle Obama "Donald Trump n'est pas le bon président pour notre pays", a déclaré l'ancienne première dame des États-Unis dans un message enregistré diffusé lors de la convention des démocrates. Les membres du parti républicain de M. Trump, qui ont été mécontents, l'ont également i

In [36]:
import torch

device = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
model_pegasus.to(device).eval()

text = """ Dear Writers,
LLMs Research publication was created to craft high-quality large language model-related articles. We genuinely want to help the LLMs community by creating useful tutorials, the latest tools, new models, repositories, and the implementation of new research papers.
In this mission, we need your support. If you’ve decided to trust us to share your article then we will make sure that your article is of high quality and truly proposes a value to the community.
How to submit an article to LLMs Research publication?
We really appreciate it if you would publish it in LLMs Research. To do that, you need to provide a sample of your work. It can be anything, a medium article (published/draft), Google Docs, or any online article you’ve submitted so far.
You need to fill out this form and then we will reach out to you with the decision if you are selected then you’ll be added as an author on the LLMs Research publication and then you can publish on the LLMs Research.
Guidelines
These are the guidelines which need to be strictly followed when writing an article:
Always add a reference to images. Use original work, do not use AI to auto-create articles. Provide proper citations of other's works and make sure the content is permissible to share
Promote resources responsibly, and carefully select what you promote in your article
Articles promoting hate words, crime, or spamming the community will result in an immediate ban from the publication
Read twice after finishing the article. Make sure that it preserves a flow and readers don’t get distracted while reading the article! """
enc = tokenizer(text, truncation=True, max_length=1024, return_tensors="pt").to(device)

with torch.no_grad():
    out = model_pegasus.generate(
        **enc,
        max_new_tokens=200, min_new_tokens=32,
        num_beams=4, length_penalty=0.8, do_sample=False
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))


Welcome to Dear Writers, the University of London School of Economics and Technology (LLMs) Research publication. This is a selection of the articles you have submitted so far.


In [37]:
!pip install -q huggingface_hub transformers safetensors

from huggingface_hub import login
login()

In [38]:
save_dir = "/content/mt5-model"  # adjust as needed

model_pegasus.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
!ls -la {save_dir}


total 2295264
drwxr-xr-x 2 root root       4096 Sep  2 11:12 .
drwxr-xr-x 1 root root       4096 Sep  2 10:28 ..
-rw-r--r-- 1 root root        884 Sep  2 11:11 config.json
-rw-r--r-- 1 root root        244 Sep  2 11:11 generation_config.json
-rw-r--r-- 1 root root 2329638768 Sep  2 11:11 model.safetensors
-rw-r--r-- 1 root root        416 Sep  2 11:12 special_tokens_map.json
-rw-r--r-- 1 root root    4309802 Sep  2 11:12 spiece.model
-rw-r--r-- 1 root root      19282 Sep  2 11:12 tokenizer_config.json
-rw-r--r-- 1 root root   16350029 Sep  2 11:12 tokenizer.json


In [39]:
from huggingface_hub import create_repo
repo_id = "vatsal18/multi-lang_summay"   # your repo

# create once (no error if it already exists)
create_repo(repo_id, private=False, exist_ok=True)

# (optional) write a simple README with metadata so the Inference Widget shows up
readme = f"""---
pipeline_tag: summarization
language:
- multilingual
library_name: transformers
license: apache-2.0
tags:
- summarization
- multilingual
- seq2seq
---

# multi-lang_summay

Fine-tuned seq2seq model for multilingual abstractive summarization.

## Usage
```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

repo_id = "{repo_id}"
tok = AutoTokenizer.from_pretrained(repo_id)
mdl = AutoModelForSeq2SeqLM.from_pretrained(repo_id).eval()

text = "Paste any article (any supported language) here."
enc = tok(text, return_tensors="pt", truncation=True, max_length=1024)
with torch.no_grad():
    out = mdl.generate(**enc, max_new_tokens=128, num_beams=4, length_penalty=0.8)
print(tok.decode(out[0], skip_special_tokens=True))
"""
with open(f"{save_dir}/README.md", "w") as f:
  f.write(readme)



In [40]:

## 4) Upload the whole folder

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # optional: faster uploads

from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=save_dir,
    repo_id=repo_id,
    commit_message="Add fine-tuned multilingual summarizer",
    ignore_patterns=["*.ipynb_checkpoints*"]
)

print("Pushed to: https://huggingface.co/" + repo_id)


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/mt5-model/spiece.model       : 100%|##########| 4.31MB / 4.31MB            

  /content/mt5-model/tokenizer.json     : 100%|##########| 16.4MB / 16.4MB            

  /content/mt5-model/model.safetensors  :   0%|          |  554kB / 2.33GB            

Pushed to: https://huggingface.co/vatsal18/multi-lang_summay


In [41]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained("vatsal18/multi-lang_summay")
mdl = AutoModelForSeq2SeqLM.from_pretrained("vatsal18/multi-lang_summay").eval()

sample = "Bonjour! Ceci est un bref article d'exemple pour tester la synthèse."
enc = tok(sample, return_tensors="pt", truncation=True, max_length=1024)
import torch
with torch.no_grad():
    out = mdl.generate(**enc, max_new_tokens=128, num_beams=4, length_penalty=0.8)
print(tok.decode(out[0], skip_special_tokens=True))


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/416 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Avez-vous déjà eu des liens avec l'addiction à internet?


In [43]:
sample = """ Architecture of LLM
Large Language Model's (LLM) architecture is determined by a number of factors, like the objective of the specific model design, the available computational resources and the kind of language processing tasks that are to be carried out by the LLM. The general architecture of LLM consists of many layers such as the feed forward layers, embedding layers, attention layers. A text which is embedded inside is collaborated together to generate predictions.

Important components to influence Large Language Model architecture:

Model Size and Parameter Count
input representations
Self-Attention Mechanisms
Training Objectives
Computational Efficiency
Decoding and Output Generation
Transformer-Based LLM Model Architectures

Transformer-based models, which have revolutionized natural language processing tasks, typically follow a general architecture that includes the following components:

Transformer-Geeksforgeeks1. Input Embeddings: The input text is tokenized into smaller units, such as words or sub-words and each token is embedded into a continuous vector representation. This embedding step captures the semantic and syntactic information of the input.

2. Positional Encoding: Positional encoding is added to the input embeddings to provide information about the positions of the tokens because transformers do not naturally encode the order of the tokens. This enables the model to process the tokens while taking their sequential order into account.

3. Encoder: Based on a neural network technique, the encoder analyses the input text and creates a number of hidden states that protect the context and meaning of text data. Multiple encoder layers make up the core of the transformer architecture. Self-attention mechanism and feed-forward neural network are the two fundamental sub-components of each encoder layer.

Self-Attention Mechanism: Self-attention enables the model to weigh the importance of different tokens in the input sequence by computing attention scores. It allows the model to consider the dependencies and relationships between different tokens in a context-aware manner.
Feed-Forward Neural Network: After the self-attention step, a feed-forward neural network is applied to each token independently. This network includes fully connected layers with non-linear activation functions, allowing the model to capture complex interactions between tokens.
4. Decoder Layers: In some transformer-based models, a decoder component is included in addition to the encoder. The decoder layers enable autoregressive generation, where the model can generate sequential outputs by attending to the previously generated tokens.

5. Multi-Head Attention: Transformers often employ multi-head attention, where self-attention is performed simultaneously with different learned attention weights. This allows the model to capture different types of relationships and attend to various parts of the input sequence simultaneously.

6. Layer Normalization: Layer normalization is applied after each sub-component or layer in the transformer architecture. It helps stabilize the learning process and improves the model's ability to generalize across different inputs.

7. Output Layers: The output layers of the transformer model can vary depending on the specific task. For example, in language modeling, a linear projection followed by SoftMax activation is commonly used to generate the probability distribution over the next token."""
enc = tok(sample, return_tensors="pt", truncation=True, max_length=1024)
import torch
with torch.no_grad():
    out = mdl.generate(**enc, max_new_tokens=400, num_beams=4, length_penalty=0.8)
print(tok.decode(out[0], skip_special_tokens=True))


The Large Language Model (LLM) is a pioneering form of language computing that has revolutionised natural language processing tasks.
